# 파생변수 v3 — **노트북 A (GPU): NN 멤버만**
이전 실험들처럼 **NN을 따로 돌려 OOF/test 아티팩트로 산출** → 노트북 B(CPU)가 입력으로 받음. GPU 한도는 NN에만 사용.
산출: `oof_nn_v3.csv`·`test_nn_v3.csv`(base + 통과후보 × 시드별) + `nn_screen.json`.
실행 후 **Save Version(Save All) → GPU 끄고 → 노트북 B(CPU)** 로.

**입력(Add Input):** train.csv / test.csv / `prune_list.json`(없으면 자동).
**누수:** NN 파생(C1/C2/C3)은 행단위, enc/med/mu/sd 전부 fold-내부 train fit. test는 train fit→predict.

## 0. 설정 + DRYRUN — 작은 드라이런으로 파이프라인·저장셀 무결성부터 확인 후 풀런

In [1]:
import os,glob,json,time,warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
warnings.filterwarnings("ignore")
# ── 드라이런 스위치: True면 30k 슬라이스·1시드·짧은 epoch로 cell 전구간 통과 확인 → 원복(False) 후 풀런
DRYRUN=False
SEEDS=(42,) if DRYRUN else (42,7,2024)
EPOCHS=6 if DRYRUN else 40
PROTO_EPOCHS=4 if DRYRUN else 8
PROTO_N=15000 if DRYRUN else 40000
def find_csv(n):
    h=[p for p in glob.glob("/kaggle/input/**/*.csv",recursive=True) if os.path.basename(p)==n]; assert h,f"{n} 없음"; return sorted(h,key=len)[0]
train=pd.read_csv(find_csv("train.csv")); test=pd.read_csv(find_csv("test.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"
if DRYRUN:
    rg=np.random.default_rng(42); idx=[]
    for cls in np.unique(train[TARGET]):
        ci=train.index[train[TARGET]==cls].to_numpy(); idx.extend(rg.choice(ci,min(len(ci),15000),replace=False))
    train=train.loc[sorted(idx)].reset_index(drop=True)
    test=test.sample(min(len(test),5000),random_state=42).reset_index(drop=True)
    print(f"⚠️ DRYRUN: train={len(train)} test={len(test)} (풀런 시 DRYRUN=False)")
y=train[TARGET].astype(int).values; N=len(train); base_rate=y.mean()
def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
COL_RSN="배아 생성 주요 이유"
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
print("train",train.shape,"| base_rate=%.4f | SEEDS=%s | EPOCHS=%d"%(base_rate,SEEDS,EPOCHS))

train (256351, 69) | base_rate=0.2583 | SEEDS=(42, 7, 2024) | EPOCHS=40


## 0b. 신규 파생(C1/C2/C3, 행단위·리크안전) + 선형/NN용 원핫 전개

In [2]:
def build_new_derived(df):
    F={}
    tx =NUM(df,"이식된 배아 수").fillna(0); sto=NUM(df,"저장된 배아 수").fillna(0); emb=NUM(df,"총 생성 배아 수").fillna(0)
    ses=(df["단일 배아 이식 여부"]==1).values
    F["EL_set_type"]=np.where(~ses,0,np.where(sto.values>0,2,1)).astype("int8")
    F["FA_no_transfer"]=(tx==0).astype("int8").values
    is_current=df[COL_RSN].astype(str).str.contains("현재 시술용",na=False).values
    F["FA_nontransfer_reason"]=(~is_current).astype("int8")
    mix=NUM(df,"혼합된 난자 수").fillna(0).values; icsi=NUM(df,"미세주입된 난자 수").fillna(0).values
    iemb=NUM(df,"미세주입에서 생성된 배아 수").fillna(0).values; embv=emb.values; both=(mix>0)&(icsi>0)
    with np.errstate(divide="ignore",invalid="ignore"):
        icsi_eff=np.where(icsi>0,iemb/icsi,np.nan); conv_eff=np.where(mix>0,(embv-iemb)/mix,np.nan)
    F["IC_route_eff_diff"]=np.where(both,icsi_eff-conv_eff,np.nan)
    return pd.DataFrame(F,index=df.index)
def expand_lin(D):
    E=pd.DataFrame(index=D.index)
    E["EL_forced"]=(D["EL_set_type"]==1).astype("int8"); E["EL_elective"]=(D["EL_set_type"]==2).astype("int8")
    E["FA_no_transfer"]=D["FA_no_transfer"]; E["FA_nontransfer_reason"]=D["FA_nontransfer_reason"]; E["IC_route_eff_diff"]=D["IC_route_eff_diff"]
    return E
Etr=expand_lin(build_new_derived(train)); Ete=expand_lin(build_new_derived(test))
NN_COLS={"C1":["EL_forced","EL_elective"],"C2":["FA_no_transfer","FA_nontransfer_reason"],"C3":["IC_route_eff_diff"]}
ALL_CANDS=["C1","C2","C3"]
assert np.array_equal(expand_lin(build_new_derived(train.head(300))).fillna(-9).values, Etr.head(300).fillna(-9).values), "행단위 독립성 위반!"
print("Etr",Etr.shape,"✅ 행단위 독립성(리크 없음)")

Etr (256351, 5) ✅ 행단위 독립성(리크 없음)


## 1. NN 파이프라인 (임베딩-MLP, GPU)

In [3]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("device:",DEVICE, "| ⚠️ GPU 아님 — 이 노트북은 GPU 세션 권장" if DEVICE=="cpu" else "")
PATIENCE=6; BATCH=2048; LR=1e-3; HIDDEN=(256,128); DROPOUT=0.2
kp=glob.glob("/kaggle/input/**/prune_list.json",recursive=True)+glob.glob("prune_list.json")
KEEP=json.load(open(kp[0]))["keep"] if kp else None
def _DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def _masks(df): return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,"ICSI":NUM(df,"미세주입된 난자 수")>0,"본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def build_ratios(df):   # nn ratio 정본 (grid nn-fe와 동일, r_ 접두)
    Mk=_masks(df); F={}
    P1=_DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=_DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=_DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["r_P1"]=P1; F["r_P2"]=P2; F["r_P6"]=P6; F["r_L3"]=L3
    F["r_P3"]=_DIV(NUM(df,"이식된 배아 수")+NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수")); F["r_P5"]=_DIV(NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수"))
    F["r_g신선수정"]=P1.where(Mk["신선"]); F["r_gICSI"]=P2.where(Mk["ICSI"]); F["r_g본인수율"]=P6.where(Mk["본인난자"]); F["r_g신선배양"]=L3.where(Mk["신선"])
    F["r_FZ2"]=_DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    return pd.DataFrame(F,index=df.index)
RATtr_nn=build_ratios(train); RATte_nn=build_ratios(test)
def prep_global(df):
    df=df.copy()
    for c in OCC:
        if c in df: df[c]=df[c].map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].map(m)
    return df
trg=prep_global(train); tte=prep_global(test)
if KEEP is None:
    Xp=train.drop(columns=[TARGET,ID_COL]).copy()
    for c in Xp.columns:
        if not pd.api.types.is_numeric_dtype(Xp[c]): Xp[c]=pd.factorize(Xp[c].astype(str))[0]
    Xp=Xp.apply(pd.to_numeric,errors="coerce"); fp=list(Xp.columns)
    a,b,ya,yb=train_test_split(Xp,y,test_size=0.25,stratify=y,random_state=42)
    mp=lgb.train(dict(objective="binary",metric="auc",learning_rate=0.05,num_leaves=63,verbose=-1,seed=42),lgb.Dataset(a,ya),300,valid_sets=[lgb.Dataset(b,yb)],callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(0)])
    bs=roc_auc_score(yb,mp.predict(b)); rp=np.random.default_rng(0); KEEP=[]
    for c in fp:
        bp=b.copy(); bp[c]=bp[c].values[rp.permutation(len(bp))]
        if bs-roc_auc_score(yb,mp.predict(bp))>=0.0001: KEEP.append(c)
def nn_cols(extra_cols):
    keep=[c for c in KEEP if c in trg.columns and c not in (TARGET,ID_COL)]
    cat=[c for c in keep if not pd.api.types.is_numeric_dtype(trg[c])]; num=[c for c in keep if c not in cat]
    return cat,num+list(extra_cols)
def fit_fold(trdf,cat,num):                          # enc/med/mu/sd 전부 fold-train만 (리크안전)
    enc={c:{v:i+1 for i,v in enumerate(trdf[c].astype(str).fillna("NA").value_counts().index)} for c in cat}
    med={c:float(trdf[c].median()) for c in num}; mu={}; sd={}
    for c in num:
        x=trdf[c].fillna(med[c]).astype(float); mu[c]=float(x.mean()); sd[c]=float(x.std())+1e-6
    return enc,med,mu,sd
def transform(df,cat,num,enc,med,mu,sd):
    Xc=np.zeros((len(df),len(cat)),dtype=np.int64)
    for j,c in enumerate(cat): Xc[:,j]=df[c].astype(str).fillna("NA").map(enc[c]).fillna(0).astype(int).values
    Xn=np.zeros((len(df),len(num)*2),dtype=np.float32)
    for j,c in enumerate(num):
        Xn[:,2*j+1]=df[c].isna().astype(np.float32).values
        Xn[:,2*j]=((df[c].fillna(med[c]).astype(float).values-mu[c])/sd[c]).astype(np.float32)
    return Xc,Xn
class EmbMLP(nn.Module):
    def __init__(self,dims,n_num,hidden=(256,128),p=0.2):
        super().__init__(); self.embs=nn.ModuleList([nn.Embedding(d,min(50,(d+1)//2+1)) for d in dims])
        et=sum(e.embedding_dim for e in self.embs); layers=[]; din=et+n_num
        for h in hidden: layers+=[nn.Linear(din,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(p)]; din=h
        layers+=[nn.Linear(din,1)]; self.mlp=nn.Sequential(*layers)
    def forward(self,xc,xn):
        x=torch.cat([torch.cat([e(xc[:,i]) for i,e in enumerate(self.embs)],1),xn],1) if len(self.embs)>0 else xn
        return self.mlp(x).squeeze(1)
def train_nn(Xc_t,Xn_t,yt,Xc_v,Xn_v,yv,dims,epochs,patience,seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    if DEVICE=="cuda": torch.cuda.manual_seed_all(seed)
    m=EmbMLP(dims,Xn_t.shape[1],HIDDEN,DROPOUT).to(DEVICE); opt=torch.optim.Adam(m.parameters(),lr=LR,weight_decay=1e-5); lf=nn.BCEWithLogitsLoss()
    dl=DataLoader(TensorDataset(torch.as_tensor(Xc_t),torch.as_tensor(Xn_t),torch.as_tensor(yt,dtype=torch.float32)),batch_size=BATCH,shuffle=True,drop_last=True)
    Xcv=torch.as_tensor(Xc_v).to(DEVICE); Xnv=torch.as_tensor(Xn_v).to(DEVICE); best=-1; bstate=None; bad=0
    for ep in range(epochs):
        m.train()
        for xc,xn,yb_ in dl:
            xc,xn,yb_=xc.to(DEVICE),xn.to(DEVICE),yb_.to(DEVICE); opt.zero_grad(); lf(m(xc,xn),yb_).backward(); opt.step()
        m.eval()
        with torch.no_grad(): auc=roc_auc_score(yv,torch.sigmoid(m(Xcv,Xnv)).cpu().numpy())
        if auc>best+1e-5: best=auc; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: break
    m.load_state_dict(bstate); return m,best
def nn_proto(extra_cols,extra_df,seed=42):
    cat,num=nn_cols(extra_cols); g=pd.concat([trg,extra_df],axis=1)
    sz=min(PROTO_N,N)
    sub=np.arange(N) if sz>=N else train_test_split(np.arange(N),train_size=sz,stratify=y,random_state=seed)[0]
    sd_=g.iloc[sub].reset_index(drop=True); ys=y[sub]
    tri,vai=train_test_split(np.arange(len(sd_)),test_size=0.2,stratify=ys,random_state=seed)
    enc,med,mu,sd=fit_fold(sd_.iloc[tri],cat,num)
    a=train_nn(*transform(sd_.iloc[tri],cat,num,enc,med,mu,sd),ys[tri],*transform(sd_.iloc[vai],cat,num,enc,med,mu,sd),ys[vai],[len(enc[c])+1 for c in cat],PROTO_EPOCHS,3,seed)[1]
    return a
def nn_full(extra_cols,extra_tr_df,extra_te_df,seed=42):
    cat,num=nn_cols(extra_cols); g=pd.concat([trg,extra_tr_df],axis=1); gte=pd.concat([tte,extra_te_df],axis=1)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(g,y)); oof=np.zeros(N); tst=np.zeros(len(test))
    for tri,vai in folds:
        enc,med,mu,sd=fit_fold(g.iloc[tri],cat,num); dims=[len(enc[c])+1 for c in cat]
        m,_=train_nn(*transform(g.iloc[tri],cat,num,enc,med,mu,sd),y[tri],*transform(g.iloc[vai],cat,num,enc,med,mu,sd),y[vai],dims,EPOCHS,PATIENCE,seed)
        def P(d):
            Xc,Xn=transform(d,cat,num,enc,med,mu,sd); m.eval(); out=[]
            with torch.no_grad():
                for i in range(0,len(Xc),16384): out.append(torch.sigmoid(m(torch.as_tensor(Xc[i:i+16384]).to(DEVICE),torch.as_tensor(Xn[i:i+16384]).to(DEVICE))).cpu().numpy())
            return np.concatenate(out)
        oof[vai]=P(g.iloc[vai]); tst+=P(gte)/len(folds)
    return oof,tst
print("NN 준비 | keep:",len(KEEP))

device: cuda 
NN 준비 | keep: 34


## 2. NN 프로토 스크리닝 (필터) → 통과후보만 풀런 대상
프로토(작게-먼저) 멀티시드 Δ로 'NN에 새 정보'인 후보만 가린 뒤, 그것만 풀런 — GPU 한도 절약.

In [4]:
t0=time.time(); nn_base_proto={s:nn_proto([],pd.DataFrame(index=train.index),s) for s in SEEDS}
nn_screen={}; CANDS_NN=[]
print("=== NN 프로토 스크리닝 ===")
for cand in ALL_CANDS:
    dn=[nn_proto(NN_COLS[cand],Etr[NN_COLS[cand]],s)-nn_base_proto[s] for s in SEEDS]
    mu,sd=float(np.mean(dn)),float(np.std(dn)); nn_screen[cand]=dict(delta=mu,std=sd)
    passed=(abs(mu)>sd and mu>0.0002) if len(SEEDS)>1 else (mu>0.0002)
    if passed: CANDS_NN.append(cand)
    print(f"  {cand}: protoΔ={mu:+.5f}±{sd:.5f} {'✅새정보→풀런' if passed else '— skip'}")
print("NN 풀런 대상:",CANDS_NN if CANDS_NN else "없음","| 프로토 소요 %.1f분"%((time.time()-t0)/60))

=== NN 프로토 스크리닝 ===
  C1: protoΔ=-0.00046±0.00030 — skip
  C2: protoΔ=-0.00011±0.00075 — skip
  C3: protoΔ=+0.00008±0.00035 — skip
NN 풀런 대상: 없음 | 프로토 소요 0.6분


## 3. NN 풀런 (base + 통과후보 × 시드) → 아티팩트 저장

In [5]:
t0=time.time(); cols_tr={"y":y}; cols_te={"ID":test[ID_COL].values}
for s in SEEDS:
    ob,tb=nn_full([],pd.DataFrame(index=train.index),pd.DataFrame(index=test.index),s)
    cols_tr[f"base_s{s}"]=ob; cols_te[f"base_s{s}"]=tb
    print(f"  base seed{s}: OOF AUC={roc_auc_score(y,ob):.5f}  ({(time.time()-t0)/60:.1f}분 누적)")
for cand in CANDS_NN:
    for s in SEEDS:
        of,tf=nn_full(NN_COLS[cand],Etr[NN_COLS[cand]],Ete[NN_COLS[cand]],s)
        cols_tr[f"{cand}_s{s}"]=of; cols_te[f"{cand}_s{s}"]=tf
        print(f"  {cand} seed{s}: OOF AUC={roc_auc_score(y,of):.5f}  ({(time.time()-t0)/60:.1f}분 누적)")
suffix="_DRYRUN" if DRYRUN else ""
pd.DataFrame(cols_tr).to_csv(f"/kaggle/working/oof_nn_v3{suffix}.csv",index=False)
pd.DataFrame(cols_te).to_csv(f"/kaggle/working/test_nn_v3{suffix}.csv",index=False)
json.dump({"screen":nn_screen,"cands_nn":CANDS_NN,"seeds":list(SEEDS),"dryrun":DRYRUN},open(f"/kaggle/working/nn_screen{suffix}.json","w"),ensure_ascii=False)
print(f"\n💾 oof_nn_v3{suffix}.csv / test_nn_v3{suffix}.csv / nn_screen{suffix}.json 저장 | NN 풀런 총 %.1f분"%((time.time()-t0)/60))
print("→ DRYRUN 통과 확인했으면 cell 0에서 DRYRUN=False로 바꿔 풀런, 끝나면 Save Version(Save All) → GPU 끄고 노트북 B(CPU)로.")

  base seed42: OOF AUC=0.73751  (3.9분 누적)
  base seed7: OOF AUC=0.73721  (7.6분 누적)
  base seed2024: OOF AUC=0.73757  (11.0분 누적)

💾 oof_nn_v3.csv / test_nn_v3.csv / nn_screen.json 저장 | NN 풀런 총 11.1분
→ DRYRUN 통과 확인했으면 cell 0에서 DRYRUN=False로 바꿔 풀런, 끝나면 Save Version(Save All) → GPU 끄고 노트북 B(CPU)로.


## 4. (선택) 그리드 nn 축용 4뷰 생성 — cands_nn 통과 시만

In [6]:
# ── 그리드 nn 축용 config-일관 4뷰 (nn_base/ratio/v3/ratio_v3). cands_nn 비면 base/ratio만.
if len(CANDS_NN)>0:
    print("\n=== nn 4뷰 생성 (그리드 축용, seed42) ===")
    uni=[c for cand in CANDS_NN for c in NN_COLS[cand]]; uni=list(dict.fromkeys(uni))
    EMPTY=pd.DataFrame(index=train.index); EMPTYte=pd.DataFrame(index=test.index)
    vb,vbt=nn_full([],EMPTY,EMPTYte,42)                                            # nn_base
    vr,vrt=nn_full(list(RATtr_nn.columns),RATtr_nn,RATte_nn,42)                    # nn_ratio
    vv,vvt=nn_full(uni,Etr[uni],Ete[uni],42)                                       # nn_v3
    e_tr=pd.concat([RATtr_nn,Etr[uni]],axis=1); e_te=pd.concat([RATte_nn,Ete[uni]],axis=1)
    vrv,vrvt=nn_full(list(e_tr.columns),e_tr,e_te,42)                              # nn_ratio_v3
    pd.DataFrame({"nn_base":vb,"nn_ratio":vr,"nn_v3":vv,"nn_ratio_v3":vrv,"y":y}).to_csv(f"/kaggle/working/oof_nn_views{suffix}.csv",index=False)
    pd.DataFrame({"ID":test[ID_COL].values,"nn_base":vbt,"nn_ratio":vrt,"nn_v3":vvt,"nn_ratio_v3":vrvt}).to_csv(f"/kaggle/working/test_nn_views{suffix}.csv",index=False)
    from sklearn.metrics import roc_auc_score
    print("  nn 뷰 OOF:",{k:round(roc_auc_score(y,v),5) for k,v in [("base",vb),("ratio",vr),("v3",vv),("ratio_v3",vrv)]})
    print(f"  💾 oof_nn_views{suffix}.csv / test_nn_views{suffix}.csv → 그리드 nn 축(커밋 nn override)")
    print("  ※ config 일관성 위해 prune_list.json(keep-33)을 입력하면 운영 nn과 정합 ↑")
else:
    print("\nNN 프로토 스크리닝 전부 미달(cands_nn 없음) → nn 4뷰 생성 skip. 그리드 nn 축은 커밋 base/ratio 사용.")



NN 프로토 스크리닝 전부 미달(cands_nn 없음) → nn 4뷰 생성 skip. 그리드 nn 축은 커밋 base/ratio 사용.
